In [1]:
!pip install langgraph langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.9 MB/s eta 0:00:00


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="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",
    model="llama-3.1-8b-instant",
    temperature=0
)

In [3]:
from typing import TypedDict, List

class ChatState(TypedDict):
    messages: List[str]
    name: str
    preferences: str

In [4]:
def chatbot(state: ChatState) -> dict:
    user_msg = state["messages"][-1]

    name = state.get("name", "")
    prefs = state.get("preferences", "")

    if "my name is" in user_msg.lower():
        name = user_msg.lower().split("my name is")[-1].strip().title()

    if "i like" in user_msg.lower():
        prefs = user_msg.lower().split("i like")[-1].strip()

    context = f"The user's name is {name}. They like {prefs}."

    reply = llm.invoke(
        context + "\nReply to: " + user_msg
    ).content

    return {
        "messages": state["messages"] + [reply],
        "name": name,
        "preferences": prefs
    }

In [5]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

In [ ]:
state = {
    "messages": [],
    "name": "",
    "preferences": ""
}

while True:
    user = input("You: ")

    if user.lower() == "quit":
        break

    state["messages"].append(user)

    state = graph.invoke(state)

    print("Bot:", state["messages"][-1])